# Audiobook Enhancer — Colab Bootstrap

This notebook is a bootstrapper only. It never contains pipeline logic —
all processing code lives in the `audiobook_engine` package in the repo.
Edit code there, not here. See `02-Architecture.md` in the vault.

In [ ]:
# 1. Clone or pull the repo (absolute path — avoids nested-clone bugs from
# a lingering %cd in another cell's execution state)
REPO_DIR = "/content/audiobook-enhancer"
REPO_URL = "https://github.com/hasnaintypes/audiobook-enhancer.git"
import os
if os.path.isdir(REPO_DIR):
    !cd "$REPO_DIR" && git pull
else:
    !git clone "$REPO_URL" "$REPO_DIR"

In [ ]:
# 2. Install pinned dependencies
import os
os.chdir("/content/audiobook-enhancer")
!pip install -q -r config/requirements.lock
!pip install -q -e .

In [ ]:
# 3. Load Neon Object Storage credentials from Colab's Secrets manager
# (the key icon in the left sidebar) — NOT pasted into this cell. Add each
# of these as a secret there first, with "Notebook access" enabled:
#   AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_ENDPOINT_URL_S3, AWS_REGION
# Values come from audiobook-enhancer/.env.local on your own machine
# (written there by `neon deploy` — see docs/07-Decisions.md D007).
import os
from google.colab import userdata

for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_ENDPOINT_URL_S3", "AWS_REGION"]:
    os.environ[key] = userdata.get(key)

print("Neon Object Storage credentials loaded from Colab Secrets.")

In [ ]:
# 4. Sanity check: run the fast unit test suite here to confirm the clone +
# dependency install actually produced a working environment.
!pytest tests/unit -q

In [ ]:
# 5. One-off smoke test: run a short clip from a test video through the
# pipeline (extract -> analyze -> upload) to confirm the environment works
# end-to-end. Not a run against the real sample set -- swap URLS for that.
#
# extract_audio() in the engine always pulls the full video, so for a quick
# test here we download only a CLIP_SECONDS-long window via yt-dlp's
# download_ranges instead of calling extract_audio() directly -- this is
# diagnostic-script logic for this cell only, not a change to the engine.
# The window is centered on the video's midpoint rather than the start.
script = '''
from pathlib import Path
import yt_dlp
from audiobook_engine.analysis.io import load_mono
from audiobook_engine.analysis.quality import analyze_quality
from audiobook_engine.analysis.strategy import select_strategy
from audiobook_engine.jobstore.storage import upload_file, object_key, presigned_url

URLS = [
    "https://youtu.be/XN9CVvZFkts",
]
CLIP_SECONDS = 60
OUTPUT_DIR = Path("/content/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def extract_middle_clip(url, output_dir, clip_seconds):
    probe_opts = {"quiet": True, "no_warnings": True, "noplaylist": True}
    with yt_dlp.YoutubeDL(probe_opts) as ydl:
        probe_info = ydl.extract_info(url, download=False)
    duration = float(probe_info.get("duration") or 0.0)

    midpoint = duration / 2
    start = max(0.0, midpoint - clip_seconds / 2)
    end = min(duration, start + clip_seconds) if duration else start + clip_seconds

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": str(output_dir / "%(id)s.%(ext)s"),
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
        "download_ranges": yt_dlp.utils.download_range_func(None, [(start, end)]),
        "force_keyframes_at_cuts": True,
        "noplaylist": True,
        "quiet": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
    video_id = info["id"]
    audio_path = output_dir / f"{video_id}.wav"
    return audio_path, info.get("title", "unknown"), duration, start, end

for i, url in enumerate(URLS, start=1):
    job_id = f"sample-{i:03d}"
    print("=" * 80)
    print(url, "->", job_id, f"({CLIP_SECONDS}s clip from the middle)")
    try:
        audio_path, title, duration, start, end = extract_middle_clip(url, OUTPUT_DIR, CLIP_SECONDS)
        print(f"Title: {title}")
        print(f"Full video duration: {duration:.1f}s -- clipped [{start:.1f}s, {end:.1f}s]")

        samples, sr = load_mono(audio_path)
        report = analyze_quality(samples, sr)
        decision = select_strategy(report)
        print(report)
        print(decision)

        key = object_key(job_id, "original", audio_path.name)
        upload_file(audio_path, key)
        print(f"Uploaded to Neon Object Storage: {key}")

        url_out = presigned_url(key, expires_in=600)
        print(f"Presigned URL (10 min): {url_out[:80]}...")
    except Exception as e:
        print(f"Failed: {e}")
'''
with open("/content/smoke_test_clip.py", "w") as f:
    f.write(script)

!python /content/smoke_test_clip.py

In [ ]:
# 6. Check sample-004 against the new detectors (background music,
# reverb, compression artifacts) -- this sample came back "clean" on
# the old metrics despite sounding bad by ear.
script = '''
from pathlib import Path
from audiobook_engine.ingestion.youtube import extract_audio, ExtractionError
from audiobook_engine.analysis.io import load_mono
from audiobook_engine.analysis.quality import analyze_quality

URL = "https://youtu.be/mJf-Wm1iJBY"  # sample-004, "clean" under the old metrics
OUTPUT_DIR = Path("/content/samples")

try:
    extracted = extract_audio(URL, OUTPUT_DIR)
    print(f"Title: {extracted.title}")

    samples, sr = load_mono(extracted.audio_path)
    report = analyze_quality(samples, sr)

    print(f"has_background_music:       {report.has_background_music} (confidence={report.background_music_confidence:.3f})")
    print(f"has_reverb:                 {report.has_reverb} (tail_score={report.reverb_tail_score:.3f})")
    print(f"has_compression_artifacts:  {report.has_compression_artifacts} (spectral_cutoff_hz={report.spectral_cutoff_hz:.1f})")
except ExtractionError as e:
    print(f"Extraction failed [{e.kind.value}]: {e}")
'''
with open("/content/check_sample_004.py", "w") as f:
    f.write(script)

!python /content/check_sample_004.py

In [ ]:
# 7. Stress-test RNNoise's -ar flag for the same end-of-stream hang that
# hit Tier 0's combined filter+resample ffmpeg call. Only the 2-minute
# clips tested so far never triggered it -- this uses a real 10-minute
# clip to push harder on the same code path, with a timeout so a real
# hang shows up as a TimeoutExpired instead of blocking the kernel.
script = """
import subprocess
import time
from pathlib import Path
import yt_dlp

from audiobook_engine.enhancement.rnnoise import download_default_model, build_ffmpeg_command

URL = "https://youtu.be/XN9CVvZFkts"
CLIP_SECONDS = 600  # 10 minutes -- stress test, not the earlier 60s smoke test
OUTPUT_DIR = Path("/content/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = Path("/content/models/std.rnnn")

model_path = download_default_model(MODEL_PATH)
print(f"Model ready at {model_path}")

probe_opts = {"quiet": True, "no_warnings": True, "noplaylist": True}
with yt_dlp.YoutubeDL(probe_opts) as ydl:
    probe_info = ydl.extract_info(URL, download=False)
duration = float(probe_info.get("duration") or 0.0)
midpoint = duration / 2
start = max(0.0, midpoint - CLIP_SECONDS / 2)
end = min(duration, start + CLIP_SECONDS)
print(f"Full duration: {duration:.1f}s -- stress clip [{start:.1f}s, {end:.1f}s]")

ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": str(OUTPUT_DIR / "%(id)s_stress.%(ext)s"),
    "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
    "download_ranges": yt_dlp.utils.download_range_func(None, [(start, end)]),
    "force_keyframes_at_cuts": True,
    "noplaylist": True,
    "quiet": True,
    "no_warnings": True,
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(URL, download=True)
input_path = OUTPUT_DIR / f"{info['id']}_stress.wav"
output_path = OUTPUT_DIR / f"{info['id']}_stress_denoised.wav"
print(f"Downloaded clip: {input_path} ({input_path.stat().st_size} bytes)")

command = build_ffmpeg_command(input_path, output_path, model_path, sample_rate=48000)
print("Running:", " ".join(command))

start_time = time.monotonic()
try:
    result = subprocess.run(command, check=True, capture_output=True, text=True, timeout=120)
    elapsed = time.monotonic() - start_time
    print(f"Completed in {elapsed:.1f}s -- no hang. Output: {output_path} ({output_path.stat().st_size} bytes)")
except subprocess.TimeoutExpired:
    elapsed = time.monotonic() - start_time
    print(f"HUNG: did not complete within {elapsed:.1f}s timeout -- reproduces the Tier 0-style hang")
except subprocess.CalledProcessError as e:
    print(f"ffmpeg failed (exit {e.returncode}): {e.stderr[-2000:]}")
"""
with open("/content/stress_test_rnnoise.py", "w") as f:
    f.write(script)

!python /content/stress_test_rnnoise.py

In [ ]:
# 8. Run the new per-chunk job runner (audiobook_engine.runner.job_runner)
# end-to-end against a real long sample -- long enough (>600s, the default
# ChunkingConfig.max_chunk_duration_s) to actually produce multiple chunks,
# not just exercise the single-chunk "end_of_audio" path.
script = """
from pathlib import Path
import yt_dlp

from audiobook_engine.runner.job_runner import run_job

URL = "https://youtu.be/XN9CVvZFkts"
CLIP_SECONDS = 1200  # 20 minutes -- long enough to force at least 2-3 chunks
OUTPUT_DIR = Path("/content/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/content/job_work")
JOB_ID = "runner-smoke-001"

probe_opts = {"quiet": True, "no_warnings": True, "noplaylist": True}
with yt_dlp.YoutubeDL(probe_opts) as ydl:
    probe_info = ydl.extract_info(URL, download=False)
duration = float(probe_info.get("duration") or 0.0)
start = 0.0
end = min(duration, CLIP_SECONDS)
print(f"Full duration: {duration:.1f}s -- runner smoke clip [{start:.1f}s, {end:.1f}s]")

ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": str(OUTPUT_DIR / "%(id)s_runner.%(ext)s"),
    "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
    "download_ranges": yt_dlp.utils.download_range_func(None, [(start, end)]),
    "force_keyframes_at_cuts": True,
    "noplaylist": True,
    "quiet": True,
    "no_warnings": True,
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(URL, download=True)
input_path = OUTPUT_DIR / f"{info['id']}_runner.wav"
print(f"Downloaded clip: {input_path} ({input_path.stat().st_size} bytes)")

manifest = run_job(JOB_ID, input_path, WORK_DIR)
print(f"Job {manifest.job_id}: {len(manifest.completed_indices)}/{manifest.total_chunks} chunks completed")
print(f"Completed indices: {sorted(manifest.completed_indices)}")
"""
with open("/content/run_job_smoke_test.py", "w") as f:
    f.write(script)

!python /content/run_job_smoke_test.py

In [ ]:
# 9. Real deliberate-interruption test for the job runner (docs/06-Roadmap.md
# Phase 4's and the whole PRD's stated success metric): simulate the process
# dying mid-job via a precisely-timed fault injection on a real clip, confirm
# the persisted manifest reflects only truly-completed chunks, then resume
# the SAME job_id and confirm completed chunks are never reprocessed while
# the chunk that crashed before its completion was persisted is safely redone.
script = """
from pathlib import Path
from collections import Counter
import yt_dlp

import audiobook_engine.jobstore.storage as storage_module
import audiobook_engine.postprocess.dsp as dsp_module
import audiobook_engine.enhancement.rnnoise as rnnoise_module
from audiobook_engine.runner.job_runner import run_job

URL = "https://youtu.be/XN9CVvZFkts"
CLIP_SECONDS = 1800  # 30 minutes -- long enough for several chunks at the 600s default
OUTPUT_DIR = Path("/content/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/content/job_work_interrupt")
JOB_ID = "interruption-test-001"

probe_opts = {"quiet": True, "no_warnings": True, "noplaylist": True}
with yt_dlp.YoutubeDL(probe_opts) as ydl:
    probe_info = ydl.extract_info(URL, download=False)
duration = float(probe_info.get("duration") or 0.0)
start = 1200.0  # a different window than the earlier smoke-test clip
end = min(duration, start + CLIP_SECONDS)
print(f"Full duration: {duration:.1f}s -- interruption-test clip [{start:.1f}s, {end:.1f}s]")

ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": str(OUTPUT_DIR / "%(id)s_interrupt.%(ext)s"),
    "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
    "download_ranges": yt_dlp.utils.download_range_func(None, [(start, end)]),
    "force_keyframes_at_cuts": True,
    "noplaylist": True,
    "quiet": True,
    "no_warnings": True,
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(URL, download=True)
input_path = OUTPUT_DIR / f"{info['id']}_interrupt.wav"
print(f"Downloaded clip: {input_path} ({input_path.stat().st_size} bytes)")

# Instrumentation: log every real chunk-processing call by its input filename,
# across BOTH the crashed run and the resumed run, so we can prove afterward
# which chunks were processed exactly once (correctly not reprocessed) vs.
# processed twice (correctly redone, because their completion was never
# persisted before the simulated crash).
processing_log = []
real_tier0 = dsp_module.run_tier0_processing
real_tier1 = rnnoise_module.run_tier1_processing

def logging_tier0(input_path, output_path, decision, sample_rate):
    processing_log.append(input_path.name)
    return real_tier0(input_path, output_path, decision, sample_rate)

def logging_tier1(input_path, output_path, model_path, sample_rate):
    processing_log.append(input_path.name)
    return real_tier1(input_path, output_path, model_path, sample_rate)

dsp_module.run_tier0_processing = logging_tier0
rnnoise_module.run_tier1_processing = logging_tier1

# Simulate a real crash: let the 2nd upload_manifest call succeed (chunk 0's
# completion durably persisted), then make the 3rd call raise instead of
# persisting -- simulating the process dying while chunk 1 is being
# finished, BEFORE its completion checkpoint is written.
real_upload_manifest = storage_module.upload_manifest
call_count = {"n": 0}

def crashing_upload_manifest(manifest):
    call_count["n"] += 1
    if call_count["n"] == 3:
        raise RuntimeError("SIMULATED INTERRUPTION -- process killed before this checkpoint")
    return real_upload_manifest(manifest)

storage_module.upload_manifest = crashing_upload_manifest

print("=" * 80)
print("PHASE 1: run_job with a simulated crash partway through")
try:
    run_job(JOB_ID, input_path, WORK_DIR)
    print("UNEXPECTED: run_job completed without the simulated crash firing")
except RuntimeError as e:
    print(f"Job crashed as expected: {e}")

storage_module.upload_manifest = real_upload_manifest  # restore before checking/resuming

manifest_after_crash = storage_module.download_manifest(JOB_ID)
print(f"Manifest after crash: {sorted(manifest_after_crash.completed_indices)} of {manifest_after_crash.total_chunks} persisted complete")
print(f"Chunks processed before crash: {processing_log}")

print("=" * 80)
print("PHASE 2: resume the SAME job_id with no fault injection")
final_manifest = run_job(JOB_ID, input_path, WORK_DIR)
print(f"Final manifest: {sorted(final_manifest.completed_indices)} of {final_manifest.total_chunks} complete")
print(f"All chunks processed (combined log across both runs): {processing_log}")

counts = Counter(processing_log)
once_only = sorted(name for name, n in counts.items() if n == 1)
reprocessed = sorted(name for name, n in counts.items() if n > 1)
print(f"Processed exactly once (correctly NOT reprocessed): {once_only}")
print(f"Processed more than once (correctly redone -- crashed before their completion was persisted): {reprocessed}")
"""
with open("/content/interruption_test.py", "w") as f:
    f.write(script)

!python /content/interruption_test.py

In [ ]:
# 11. Run the new crossfade stitching (audiobook_engine.stitching.stitch)
# against the already-completed runner-smoke-001 job, confirm the uploaded
# whole-file deliverable exists with the expected (crossfade-adjusted)
# duration -- mechanical verification only (duration/existence), not a
# by-ear "no audible seams" check, which is explicitly deferred.
script = """
from pathlib import Path
import soundfile as sf

from audiobook_engine.jobstore.storage import get_s3_client, BUCKET_NAME, object_key
from audiobook_engine.stitching.stitch import stitch_job

JOB_ID = "runner-smoke-001"
WORK_DIR = Path("/content/stitch_work")

stitched_path = stitch_job(JOB_ID, WORK_DIR, crossfade_ms=50.0)
samples, sr = sf.read(str(stitched_path))
duration_s = len(samples) / sr
print(f"Stitched file: {stitched_path} -- {duration_s:.2f}s at {sr}Hz")

client = get_s3_client()
key = object_key(JOB_ID, "enhanced", "final.wav")
head = client.head_object(Bucket=BUCKET_NAME, Key=key)
print(f"Uploaded deliverable confirmed in storage: {key} ({head['ContentLength']} bytes)")
"""
with open("/content/stitch_smoke_test.py", "w") as f:
    f.write(script)

!python /content/stitch_smoke_test.py